In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

## import

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [ ]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Note_Books")
cwd = os.getcwd()
print(cwd)

sys.path.append("../")

/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d


In [ ]:
import scanpy as sc
import spatialdata as sd
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [ ]:
import Multi_Modal_Rep as MMR

In [ ]:
import importlib
import Multi_Modal_Rep.tools
import Multi_Modal_Rep.model

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: Use

SpatialData object, with associated Zarr store: /content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 24689, 17051), (3, 12344, 8525), (3, 6172, 4262), (3, 3086, 2131), (3, 1543, 1065)
│     └── 'morphology_focus': DataTree[cyx] (4, 23912, 34154), (4, 11956, 17077), (4, 5978, 8538), (4, 2989, 4269), (4, 1494, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
│     └── 'nucleus_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (63173, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (63173, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (63036, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (63173, 5006)
with coordi

In [ ]:
adata = sdata.tables["table"]
adata

AnnData object with n_obs × n_vars = 63173 × 5006
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

(63173, 5006)

In [ ]:
adata_omiCLIP = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/cells.h5ad")
adata_omiCLIP

AnnData object with n_obs × n_vars = 63173 × 1
    obs: 'cell_id'
    obsm: 'X_custom'

In [ ]:
# 1. Ensure cell IDs are the index (not just a column)
if 'cell_id' in adata.obs.columns:
    adata.obs.set_index('cell_id', inplace=True)
if 'cell_id' in adata_omiCLIP.obs.columns:
    adata_omiCLIP.obs.set_index('cell_id', inplace=True)

# 2. Align the two objects by cell_id (intersection)
common_ids = adata.obs_names.intersection(adata_omiCLIP.obs_names)

# Optional: check how many matched
print(f"Matched {len(common_ids)} cells out of {adata.n_obs}")

# 3. Reorder both to the same order
adata_c = adata[common_ids, :].copy()
adata_omiCLIP_c = adata_omiCLIP[common_ids, :].copy()

# 4. Add the X_custom matrix to adata_main.obsm
adata_c.obsm['Morpho_Embedding'] = adata_omiCLIP_c.obsm['X_custom']

# 5. Done! Verify
print(adata_c.obsm.keys())

Matched 63173 cells out of 63173
KeysView(AxisArrays with keys: spatial, Morpho_Embedding)


In [ ]:
adata_c.write("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

# Dataset

In [ ]:
adata = sc.read_h5ad("../../Data/Breast_Cancer/ann_data.h5ad")

Founsation Models

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata.obs_names = [str(int(cid[63:])-1) for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['UNI'][edata.obs_names.get_indexer(common_cells)]

In [ ]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Preparing Dataset

In [ ]:
adatas = MMR.prep_adatas(adata, norm=True, log1p=True)
dataset = MMR.make_dataset(adatas, sparse_graph=True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)

Expression torch.Size([167780, 313])
Morpho_Embedding torch.Size([167780, 1536])
Neighborhood_Graph torch.Size([2, 1342240])
Regional_Expression torch.Size([1, 313])
Regional_Morpho torch.Size([1, 1536])
Regional_Neighbors torch.Size([2, 167780])


In [ ]:
importlib.reload(MMR.dataset)
importlib.reload(MMR.model)
importlib.reload(MMR)

<module 'steamboat_m_integrated_1_d' from '/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d/../steamboat_m_integrated_1_d/__init__.py'>

# Train

## Model Type 0

In [ ]:
model_type = 0

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 1

In [ ]:
model_type = 1

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 2

In [ ]:
model_type = 2

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 3

In [ ]:
model_type = 3

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 4

In [ ]:
model_type = 4

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 5

In [ ]:
model_type = 5

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 6

In [ ]:
model_type = 6

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 7

In [ ]:
model_type = 7

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 8

In [ ]:
model_type = 8

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 9

In [ ]:
model_type = 9

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 10

In [ ]:
model_type = 10

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 11

In [ ]:
model_type = 11

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

## Model Type 12

In [ ]:
model_type = 12

MMR.set_random_seed(0)
model = MMR.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

# Check